To run this, press "*Runtime*" and press "*Run all*" on a **free** Tesla T4 Google Colab instance!
<div class="align-center">
<a href="https://unsloth.ai/"><img src="https://github.com/unslothai/unsloth/raw/main/images/unsloth%20new%20logo.png" width="115"></a>
<a href="https://discord.gg/unsloth"><img src="https://github.com/unslothai/unsloth/raw/main/images/Discord button.png" width="145"></a>
<a href="https://docs.unsloth.ai/"><img src="https://github.com/unslothai/unsloth/blob/main/images/documentation%20green%20button.png?raw=true" width="125"></a></a> Join Discord if you need help + ⭐ <i>Star us on <a href="https://github.com/unslothai/unsloth">Github</a> </i> ⭐
</div>

To install Unsloth your local device, follow [our guide](https://docs.unsloth.ai/get-started/install-and-update). This notebook is licensed [LGPL-3.0](https://github.com/unslothai/notebooks?tab=LGPL-3.0-1-ov-file#readme).

You will learn how to do [data prep](#Data), how to [train](#Train), how to [run the model](#Inference), & [how to save it](#Save)


### News


Introducing FP8 precision training for faster RL inference. [Read Blog](https://docs.unsloth.ai/new/fp8-reinforcement-learning).

Unsloth's [Docker image](https://hub.docker.com/r/unsloth/unsloth) is here! Start training with no setup & environment issues. [Read our Guide](https://docs.unsloth.ai/new/how-to-train-llms-with-unsloth-and-docker).

[gpt-oss RL](https://docs.unsloth.ai/new/gpt-oss-reinforcement-learning) is now supported with the fastest inference & lowest VRAM. Try our [new notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/gpt-oss-(20B)-GRPO.ipynb) which creates kernels!

Introducing [Vision](https://docs.unsloth.ai/new/vision-reinforcement-learning-vlm-rl) and [Standby](https://docs.unsloth.ai/basics/memory-efficient-rl) for RL! Train Qwen, Gemma etc. VLMs with GSPO - even faster with less VRAM.

Visit our docs for all our [model uploads](https://docs.unsloth.ai/get-started/all-our-models) and [notebooks](https://docs.unsloth.ai/get-started/unsloth-notebooks).


### Installation

In [1]:
%%capture
import os, re
if "COLAB_" not in "".join(os.environ.keys()):
    !pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
    xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
    !pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
    !pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2
!git clone https://github.com/SparkAudio/Spark-TTS
!pip install omegaconf einx torchcodec "datasets>=3.4.1,<4.0.0"

In [2]:
!git clone https://github.com/SparkAudio/Spark-TTS
!pip install omegaconf einx torchcodec "datasets>=3.4.1,<4.0.0"

fatal: destination path 'Spark-TTS' already exists and is not an empty directory.


### Unsloth

`FastModel` supports loading nearly any model now! This includes Vision and Text models!

Thank you to [Etherl](https://huggingface.co/Etherll) for creating this notebook!

In [3]:
!uv pip install torchvision>=0.24.0 --system

Using Python 3.12.12 environment at: /usr
Audited 1 package in 295ms


In [4]:
!uv pip install wandb --upgrade --system

Using Python 3.12.12 environment at: /usr
Resolved 20 packages in 581ms
Prepared 7 packages in 692ms
Uninstalled 7 packages in 99ms
Installed 7 packages in 52ms
 - packaging==25.0
 + packaging==26.0
 - protobuf==5.29.5
 + protobuf==6.33.5
 - pydantic==2.12.3
 + pydantic==2.12.5
 - pydantic-core==2.41.4
 + pydantic-core==2.41.5
 - requests==2.32.4
 + requests==2.32.5
 - urllib3==2.5.0
 + urllib3==2.6.3
 - wandb==0.24.0
 + wandb==0.24.1


In [12]:
from unsloth import FastModel
import torch
from huggingface_hub import snapshot_download

max_seq_length = 2048 # Choose any for long context!

fourbit_models = [
    # 4bit dynamic quants for superior accuracy and low memory use
    "unsloth/gemma-3-4b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-12b-it-unsloth-bnb-4bit",
    "unsloth/gemma-3-27b-it-unsloth-bnb-4bit",
    # Qwen3 new models
    "unsloth/Qwen3-4B-unsloth-bnb-4bit",
    "unsloth/Qwen3-8B-unsloth-bnb-4bit",
    # Other very popular models!
    "unsloth/Llama-3.1-8B",
    "unsloth/Llama-3.2-3B",
    "unsloth/Llama-3.3-70B",
    "unsloth/mistral-7b-instruct-v0.3",
    "unsloth/Phi-4",
] # More models at https://huggingface.co/unsloth

# Download model and code
snapshot_download("MohamedGomaa30/spark-tts-normazlied-masri-mega", local_dir = "Spark-TTS-0.5B")

model, tokenizer = FastModel.from_pretrained(
    model_name = "MohamedGomaa30/spark-tts-normazlied-masri-mega",
    max_seq_length = max_seq_length,
    dtype = torch.float32, # Spark seems to only work on float32 for now
    full_finetuning = True, # We support full finetuning now!
    load_in_4bit = False,
    #token = "hf_...", # use one if using gated models like meta-llama/Llama-2-7b-hf
)

==((====))==  Unsloth 2026.1.4: Fast Qwen2 patching. Transformers: 4.56.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.741 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 7.5. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.33.post1. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: Float16 full finetuning uses more memory since we upcast weights to float32.


We now add LoRA adapters so we only need to update 1 to 10% of all parameters!

In [7]:
import locale
import torchaudio.transforms as T
import os
import torch
import sys
import numpy as np
sys.path.append('Spark-TTS')
from sparktts.models.audio_tokenizer import BiCodecTokenizer
from sparktts.utils.audio import audio_volume_normalize

<a name="Inference"></a>
### Inference
Let's run the model! You can change the prompts


In [8]:
input_text =   "الولد الصغير كان بيجري في الشارع وبيضحك بصوت عالي جدًا كأنه لسه كسبان في لعبة"
chosen_voice = None # None for single-speaker

In [23]:
#@title Final Fixed Inference Code

import torch
import re
import numpy as np
import soundfile as sf
from typing import Dict, Any
from IPython.display import Audio, display

@torch.inference_mode()
def generate_speech_from_text(
    text: str,
    temperature: float = 0.7,
    top_k: int = 50,
    top_p: float = 1.0,
    max_new_audio_tokens: int = 2048,
    device: torch.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
) -> np.ndarray:
    """
    Generates speech by extracting exactly 32 global tokens to match the speaker encoder.
    """
    torch.compiler.reset()

    # Match prompt format to your specific preprocessing
    prompt = f"<|task_tts|><|start_content|>{text}<|end_content|><|start_global_token|>"
    model_inputs = tokenizer([prompt], return_tensors="pt").to(device)

    print("Generating token sequence...")
    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=max_new_audio_tokens,
        do_sample=True,
        temperature=temperature,
        top_k=top_k,
        top_p=top_p,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id
    )

    # 1. Decode generated content
    generated_ids_trimmed = generated_ids[:, model_inputs.input_ids.shape[1]:]
    predicts_text = tokenizer.batch_decode(generated_ids_trimmed, skip_special_tokens=False)[0]

    # 2. Extract Semantic Tokens
    semantic_matches = re.findall(r"<\|bicodec_semantic_(\d+)\|>", predicts_text)
    if not semantic_matches:
        print("Warning: No semantic tokens found.")
        return np.array([], dtype=np.float32)
    pred_semantic_ids = torch.tensor([int(t) for t in semantic_matches]).long().unsqueeze(0).to(device)

    # 3. Extract and Force exactly 32 Global Tokens
    global_matches = re.findall(r"<\|bicodec_global_(\d+)\|>", predicts_text)
    global_ids_list = [int(t) for t in global_matches]

    # CRITICAL FIX: The Spark-TTS 0.5B project layer expects exactly 32 tokens (4096 dim)
    expected_count = 32
    if len(global_ids_list) < expected_count:
        # Pad with the last token or 0 if missing
        pad_val = global_ids_list[-1] if global_ids_list else 0
        global_ids_list += [pad_val] * (expected_count - len(global_ids_list))
    else:
        global_ids_list = global_ids_list[:expected_count]

    # Shape must be (1, 32)
    pred_global_ids = torch.tensor(global_ids_list).long().unsqueeze(0).to(device)

    print(f"Detokenizing: Global {pred_global_ids.shape}, Semantic {pred_semantic_ids.shape}")

    # 4. Detokenize using BiCodec
    audio_tokenizer.model.to(device)
    # The .detokenize method internally handles the required .unsqueeze(1) for global tokens
    wav_np = audio_tokenizer.detokenize(
        pred_global_ids,
        pred_semantic_ids
    )

    return wav_np

if __name__ == "__main__":
    # Arabic input example
    input_text = "و بعدين يا عزيزي"

    generated_waveform = generate_speech_from_text(input_text)

    if generated_waveform.size > 0:
        output_filename = "final_output.wav"
        sample_rate = audio_tokenizer.config.get("sample_rate", 16000)
        sf.write(output_filename, generated_waveform, sample_rate)
        print(f"Success! Audio saved to {output_filename}")
        display(Audio(generated_waveform, rate=sample_rate))
    else:
        print("Generation failed.")


Generating token sequence...
Detokenizing: Global torch.Size([1, 32]), Semantic torch.Size([1, 60])
Success! Audio saved to final_output.wav
